# LAB 2 &ndash; Shuffling cards

In this second lab, you'll get acquainted with AES and learn how to cryptographically shuffle a deck of cards.

First thing to do: double-click on this text cell to write your <b style="color: red">HACKOOLIQUES</b>.

## 0) Bytes

Byte (« octets ») arrays in Python 3 can be thought of a list of integers between 0 and 255: 

In [16]:
a = bytes([72, 69, 76, 76, 79])

a

b'HELLO'

The corresponding ASCII characters are displayed for convenience (the `b` reminds us that it is not a regular string but a byte array), but the individual bytes should not be thought of as "characters" (integers are returned):

In [ ]:
for c in a:
    print(c)

72
69
76
76
79


: 

Also, be aware that most values don't correspond to printable ASCII characters and just won't display nicely:

In [ ]:
b = bytes([145, 8, 203, 78, 23])
b

b'\x91\x08\xcbN\x17'

: 

Hexadecimal values are displayed for bytes for which there is no better option; in general, this is the preferred way to look at a byte array (2 hexadecimal numbers for each byte).

In [ ]:
b.hex()

'9108cb4e17'

: 

It's easy to implement the XOR (or $\oplus$) operation for bytes since Python has a built-in bitwise-XOR operator for integers.

In [ ]:
def xor(a: bytes, b: bytes):
    return bytes([x^y for x,y in zip(a,b)])

print("a      :", a.hex())
print("b      :", b.hex())
print("a xor b:", xor(a,b).hex())

a      : 48454c4c4f
b      : 9108cb4e17
a xor b: d94d870258


: 

**To do:** Before you proceed, make sure everything above makes sense to you (experience shows that you will probably need to read it again in a couple of minutes)...

As a warm-up, make sure you are able to turn a string of hex digits (such as `'48454c4c4f'`) into a `bytes` object (such as `a`).

In [ ]:
h = "0x48454c4c4f"
b = bytes.fromhex(h[2:])
b

b'HELLO'

: 

## 1) Using AES

We will use the implementation of AES provided by the <a href="https://pypi.org/project/cryptography/">cryptography</a> library (`pip install cryptography` if needed).

In [ ]:
import os

# Hazardous Materials: use in real-world applications only if you know what you're doing!
# cryptography also provides "idiot-proof" Recipes that should be preferred

from cryptography.hazmat.primitives.ciphers import Cipher
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.primitives.ciphers.modes import ECB
from cryptography.hazmat.backends import default_backend

: 

AES works with blocks of 128 bits, _i.e._ 16 bytes.

In [ ]:
AES.block_size # in bits

128

: 

So let us now initialize AES with a randomly generated 16-byte key. 

In [ ]:
k = os.urandom(16)

print("Secret key:", k.hex())

cipher = Cipher(AES(k), ECB(), default_backend())

encryptor = cipher.encryptor()  # E(k, ) in the slides

Secret key: 6e49171f65b400c77c41b5fbbb78b615


: 

We can now start encrypting text. 

In [ ]:
m = b"Don't forget to respect the block length. A reversible padding scheme is needed in general.#####"

encryptor = cipher.encryptor()
c = encryptor.update(m)

for i in range(len(c)//16):
    print(c[16*i:16*(i+1)].hex())

32db4cea82565a3df0dc3d5ba6316f85
582a5eeb78ce0dcbf8a888246e80b6d1
660c3e533aa2dc896f681dc3d21892eb
c47495e7a889ca8779472b4637a59a26
fbd738f4737408e8079d32734b5fd837
ab45584febe663e08188b9c4c0540651


: 

In [ ]:
decryptor = cipher.decryptor()
decryptor.update(c)

b"Don't forget to respect the block length. A reversible padding scheme is needed in general.#####"

: 

Notice what happens when blocks repeat: 

In [ ]:
m = b"Don't use ECB...Don't use ECB...Don't use ECB..."

c = encryptor.update(m)

for i in range(len(c)//16):
    print(c[16*i:16*(i+1)].hex())

71e5dba200466923039ff3f6ffbf4a89
71e5dba200466923039ff3f6ffbf4a89
71e5dba200466923039ff3f6ffbf4a89


: 

This breaks semantic security, since the attacker should not be allowed to know that the message has repeating blocks.

<b>To do</b>: Encrypt the same message using AES in cipher block chaining (CBC) mode "by hand" (<i>i.e.</i>, **don't** use the `cryptography` API except to encrypt single blocks, set your encryptor to ECB mode). Verify that Bob is able to decrypt the received ciphertext knowing only the shared secret key $k$.

In [49]:
def cbc_encrypt(k, blocks, iv):
    encryptor = Cipher(AES(k), ECB(), default_backend()).encryptor()
    c_prev = iv
    c = []
    for b in blocks:
        u = encryptor.update(xor(b, c_prev))
        c.append(u)
        c_prev = u
    return c

def cbc_decrypt(k, c):
    decryptor = Cipher(AES(k), ECB(), backend=default_backend()).decryptor()
    d = []
    for i in range(1, len(c)): 
        decrypted_block = decryptor.update(c[i]) 
        plain_block = xor(decrypted_block, c[i-1]) 
        d.append(plain_block)
    return d

def encode_utf8_pad(s):
    return s.encode('utf-8').ljust(16, b"\0")

def decode_utf8_pad(s):
    return s.rstrip(b"\0").decode('utf-8')


In [50]:
k = os.urandom(16)
print(f"Alice transmet la clef k={k.hex()} par un canal sécurisé à Bob")


m = b"Don't use ECB...Don't use ECB...Don't use ECB..."

print(f'Alice encode le message "{m}" avec la clef k')
cipher_alice = Cipher(AES(k), ECB(), default_backend())
encryptor_alice = cipher_alice.encryptor()


blocks = [m[i:i+16] for i in range(0, len(m), 16)] # découpé en blocs de 16 octets
iv = bytes.fromhex( os.urandom(16).hex() )
c = [iv] + cbc_encrypt(k, blocks, iv)

    
c_hex = [c_i.hex() for c_i in c]
print(f'Alice transmets le chiffré suivant à Bob {c_hex}')
print(f"Eve intercepte le chiffré mais en l'absence de la clef elle ne peut rien déduire du message")
print(f"Bloc1 == Bloc2, Bloc2 == Bloc3", c[1]==c[2], c[2]==c[3])
print(f'Bob en possession de la clef k {k.hex()} peut déchiffrer le message. C0 initialisera le contexte.')


d = cbc_decrypt(k, c)  
print(  b"".join(d))



Alice transmet la clef k=aa22f539c8d55a2d9457df0b2bfdb694 par un canal sécurisé à Bob
Alice encode le message "b"Don't use ECB...Don't use ECB...Don't use ECB..."" avec la clef k
Alice transmets le chiffré suivant à Bob ['e9e5a712973194fbc441617b558888b9', '5c0ce1e6d143deacbea9835be9bbd6ce', 'a7cd79fc3bc7423c643fa8da8fd516d1', '34c081eb7191b27516e131b2618b6e65']
Eve intercepte le chiffré mais en l'absence de la clef elle ne peut rien déduire du message
Bloc1 == Bloc2, Bloc2 == Bloc3 False False
Bob en possession de la clef k aa22f539c8d55a2d9457df0b2bfdb694 peut déchiffrer le message. C0 initialisera le contexte.
b"Don't use ECB...Don't use ECB...Don't use ECB..."


Le mode ECB est faible car des blocs identiques donnent toujours le même chiffré, révélant des motifs dans le message. CBC corrige cela en introduisant un IV aléatoire et en chiffrant chaque bloc avec le XOR du bloc clair et du bloc chiffré précédent : ci = E(k, mi ⊕ ci-1). Ainsi chaque bloc identique produit un résultat différent selon le contexte, renforçant la sécurité.

## 2) Shuffling cards, pt. 1

Let's turn to a seemingly unrelated problem: that of a casino wanting to "impredictably" (from the point of view of the players) but "reproducibly" (from its point of view) shuffle a deck of digital playing cards with a 128-bit shuffle key (probably generated from a master key using a CSPRNG). There are $52! \approx 2^{226}$ different ways to shuffle a deck of cards, so the whole description of a shuffle couldn't possibly fit in the shuffle key: it has to be securely derived from it.

The first, "easiest" way to do it would be to:

1. Encrypt all card values using the shuffle key.
2. Sort lexicographically the resulting ciphertexts.
3. Assign to every "plaincard" its position in the sorted "ciphercards" list.

Hence obtaining a shuffled list of the original cards.

<b>To do</b>: Do it! What is the first poker hand (first 5 cards) dealt from your shuffled card deck?

# CBC
c0 = random Initial Value
ci = E (k , m[i] ⊕ c[i −1])

In [ ]:
from cryptography.hazmat.primitives.ciphers import Cipher
from cryptography.hazmat.primitives.ciphers.algorithms import AES
from cryptography.hazmat.primitives.ciphers.modes import ECB
from cryptography.hazmat.backends import default_backend
from os import urandom

k = urandom(16)
iv = urandom(16)

ranks = ["2","3","4","5","6","7","8","9","10","J","Q","K","A"]
suits = ["♣","♦","♥","♠"]
deck = [r+s for s in suits for r in ranks]

encoded_deck = [encode_utf8_pad(c) for c in deck]
encrypted_deck = cbc_encrypt(k, encoded_deck, iv)
shuffled_encrypted = sorted(encrypted_deck)
shuffled_orders = [shuffled_encrypted.index(card) for card in encrypted_deck]

first_hand = [ deck[i] for i in shuffled_orders[:5]]
first_hand


['3♦', '7♥', 'K♠', 'K♦', '3♠']

Le code chiffre chaque carte avec AES en mode CBC puis trie les résultats pour créer un ordre de cartes à la fois imprévisible pour les joueurs et reproductible pour le casino grâce à la clé. Cette méthode assure que le deck est mélangé de façon sécurisée et que les cinq premières cartes peuvent être distribuées de manière fiable tout en empêchant quiconque sans la clé de deviner l’ordre.

## 3) Shuffling cards, pt. 2

The above method works well, but rapidly becomes incovenient when the number of objects to permute gets large. Suppose for example that we want to shuffle not 52, but 52000 cards: with the above method, getting the top 5 would require 52000 single AES evaluations. A more efficient solution (look up  "Format-preserving encryption" for more information) is the following:

<ol>
    <li>Write every card value as a 2-byte word.</li>
    <li>Given a card value $m$, apply to it 3 rounds of a Feistel network with inner function $f$, where        
        $ f(b) = $ first byte of the AES encryption of byte $b$ padded with 15 # signs.</li> 
    <li>Repeat step 2. until the resulting value falls in the $[0,51999]$ range; this is the value of the card at position $m$ after the permutation.
</ol>

<b>To do:</b> What are the 5 first cards we get out of this 52000-card deck? How many AES evaluations did this take?

In [80]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from os import urandom

AES_CPT = 0

def aes_encrypt(block, k):
    global AES_CPT
    AES_CPT += 1
    cipher = Cipher(algorithms.AES(k), modes.ECB(), backend=default_backend())
    encryptor = cipher.encryptor()
    return encryptor.update(block)

def feistel(word, k, upperbound=52000, loops=3):
    def f(b):
        block = bytes([b]) + b"#"*15
        return aes_encrypt(block, k)[0]

    left, right = word[0], word[1]
    for _ in range(loops):
        left, right = right, left ^ f(right)

    result = (left << 8) | right
    if result >= upperbound:
        new_word = bytes([left, right])
        return feistel(new_word, k, upperbound=upperbound, loops=loops)
    return bytes([left, right])

def main(k, deck_size=52000, top_n=5):
    global AES_CPT
    AES_CPT = 0
    
    first_cards = []
    for m in range(top_n):
        word = bytes([m >> 8, m & 0xFF])
        shuffled_word = feistel(word, k, upperbound=deck_size)
        shuffled_index = (shuffled_word[0] << 8) | shuffled_word[1]
        first_cards.append(shuffled_index)
    return first_cards, AES_CPT

k = urandom(16)
print("Clef secrète:", k.hex())
top_5_1, total_aes_1 = main(k, 52000,5)
print("Indices des 5 premières cartes :", top_5_1, f"({total_aes_1} évals)")
print("Relance du programme avec la même clef:")
top_5_2, total_aes_2 = main(k, 52000,5)
print("Indices des 5 premières cartes :", top_5_2, f"({total_aes_2} évals)")
print("Résultats égaux :", top_5_1 == top_5_2 and total_aes_1 == total_aes_2)
    


Clef secrète: d2ae57cb582fb8bd466d430b7d54a0bf
Indices des 5 premières cartes : [38856, 36492, 26297, 50162, 19397] (24 évals)
Relance du programme avec la même clef:
Indices des 5 premières cartes : [38856, 36492, 26297, 50162, 19397] (24 évals)
Résultats égaux : True


Avec le mode CTR et compteur randomisé, chaque bloc est chiffré indépendamment, ce qui autorise la parallélisation, et le compteur aléatoire empêche la répétition des séquences pour des messages identiques, renforçant la sécurité. 

CBC est sûr mais séquentiel et lent sur de grands decks (toutes les cartes doivent être chiffrées), alors que CTR randomized permet un accès direct, rapide et parallèle, parfait pour un casino en ligne qui doit distribuer des cartes rapidement et de façon imprévisible.